# Artificial Intelligence Agents — Laboratory Submission

## Goal-Based Warehouse Navigation Agent

This notebook contains the **implementation, execution, and tests** for the warehouse-navigation laboratory.

**The conceptual/reflection answers are intentionally not included here; they are in the accompanying Word report.**

## Warehouse environment

```text
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################
```

`S` = start, `G` = goal, `#` = obstacle, `.` = free space.

Allowed actions: **Up, Down, Left, Right**.

In [ ]:
from collections import deque

warehouse = [
    "#####################",
    "#S....#............G#",
    "#.##....##########..#",
    "#....##.............#",
    "#.######.###.#.###..#",
    "#........#..........#",
    "#####################",
]

ACTIONS = {
    "Up": (-1, 0),
    "Down": (1, 0),
    "Left": (0, -1),
    "Right": (0, 1),
}

def bfs_path(grid):
    start = next(( (r,c) for r,row in enumerate(grid)
                   for c,ch in enumerate(row) if ch=="S" ), None)
    goal = next(( (r,c) for r,row in enumerate(grid)
                  for c,ch in enumerate(row) if ch=="G" ), None)

    queue = deque([start])
    parent = {start: None}
    action_used = {}

    while queue:
        current = queue.popleft()
        if current == goal:
            break

        for action, (dr,dc) in ACTIONS.items():
            nxt = (current[0]+dr, current[1]+dc)

            if (0 <= nxt[0] < len(grid)
                and 0 <= nxt[1] < len(grid[0])
                and grid[nxt[0]][nxt[1]] != "#"
                and nxt not in parent):
                parent[nxt] = current
                action_used[nxt] = action
                queue.append(nxt)

    if goal not in parent:
        return None, None

    path, actions = [], []
    current = goal
    while current is not None:
        path.append(current)
        if current in action_used:
            actions.append(action_used[current])
        current = parent[current]

    return path[::-1], actions[::-1]

In [ ]:
path, actions = bfs_path(warehouse)

print("Path:")
print(path)
print()
print("Actions:")
print(actions)
print()
print("Number of moves:", len(actions))
print("Start:", path[0])
print("Goal:", path[-1])

In [ ]:
# Validation tests

assert path is not None
assert path[0] == (1, 1)
assert path[-1] == (1, 19)

# No obstacle collisions.
for r, c in path:
    assert warehouse[r][c] != "#"

# Every action is one legal grid-square move.
for i, action in enumerate(actions):
    dr, dc = ACTIONS[action]
    assert path[i + 1] == (path[i][0] + dr, path[i][1] + dc)

# BFS gives a shortest path for this unweighted grid.
assert len(actions) == 20

print("All path-validation tests passed.")

In [ ]:
# Failure-case test: block both exits from S.

blocked = [list(row) for row in warehouse]
blocked[2][1] = "#"
blocked[1][2] = "#"
blocked_grid = ["".join(row) for row in blocked]

no_path, no_actions = bfs_path(blocked_grid)

assert no_path is None
assert no_actions is None

print("No-path test passed.")

In [ ]:
# Compact human-readable route output.

if path is None:
    print("No collision-free path exists.")
else:
    print("Route:")
    print(" -> ".join(map(str, path)))
    print("\nActions:")
    print(" -> ".join(actions))